In [17]:
import pandas as pd 

mises_citations_df = pd.read_csv("../data/processed/citations_with_mises_part.csv")

In [18]:
mises_citations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266018 entries, 0 to 266017
Data columns (total 21 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Unnamed: 0                266018 non-null  int64  
 1   paper_id                  266018 non-null  int64  
 2   raw                       265936 non-null  object 
 3   context                   266018 non-null  object 
 4   co_cited_count            266018 non-null  int64  
 5   section_id                259319 non-null  object 
 6   paragraph_id              265976 non-null  object 
 7   sentence_id               266018 non-null  object 
 8   sentence_seq_number       266018 non-null  int64  
 9   reference_seq_number      266018 non-null  int64  
 10  author                    227920 non-null  object 
 11  page                      74695 non-null   float64
 12  year                      223130 non-null  float64
 13  title                     266018 non-null  o

In [19]:
mises_citations_df.head()

,Unnamed: 0,paper_id,raw,context,co_cited_count,section_id,paragraph_id,sentence_id,sentence_seq_number,reference_seq_number,...,page,year,title,filename,sentence_count,reference_count,source title,similarity,human_action_part_number,mises_part
0,0,1,Streissler (1972),"In particular, as the classic contributions of...",0,_gzuY8cP,_NwgSU35,_TMX2wdg,4,1,...,NaN,1972.0,Unknown Title,10.1002.9780470999059.ch17.pdf.grobid.tei.xml,298,20,Review of Austrian Economics,44.444444,NaN,Streissler
1,1,1,"(Menger, [1981] 1871, p. 49)",The final two paragraphs of the preface are a ...,1,_NytzfyY,_Z99Bvn5,_aJz9uAb,10,2,...,49.0,1981.0,Unknown Title,10.1002.9780470999059.ch17.pdf.grobid.tei.xml,298,20,Review of Austrian Economics,44.444444,NaN,Menger
2,2,1,"1871, p. 49)",The final two paragraphs of the preface are a ...,1,_NytzfyY,_Z99Bvn5,_aJz9uAb,10,3,...,49.0,1871.0,Unknown Title,10.1002.9780470999059.ch17.pdf.grobid.tei.xml,298,20,Review of Austrian Economics,44.444444,NaN,NaN
3,3,1,"(Menger, [1981] 1871, p. 48)",He also goes to some length in the preface to ...,0,_NytzfyY,_Z99Bvn5,_54sgkMV,11,4,...,48.0,1981.0,Unknown Title,10.1002.9780470999059.ch17.pdf.grobid.tei.xml,298,20,Review of Austrian Economics,44.444444,NaN,Menger
4,4,1,Menger's (1985Menger's ( [1883] ] ),A detour into Menger's (1985Menger's ( [1883] ...,0,_NytzfyY,_ZYekVnE,_F7mv66u,16,5,...,NaN,1883.0,Unknown Title,10.1002.9780470999059.ch17.pdf.grobid.tei.xml,298,20,Review of Austrian Economics,44.444444,NaN,Menger


In [29]:
import pandas as pd

pd.set_option("display.max_rows", None)

level = "paragraph_id"

df = mises_citations_df.copy()

# 1. Mantém apenas linhas com section_id e author
df = df.dropna(subset=["author", "section_id"])

# 2. Separa citações ao Mises (que carregam a parte do Human Action)
mises_df = (
    df
    .loc[df["author"] == "Mises"]
    .dropna(subset=["mises_part"])
    [[level, "mises_part"]]
    .drop_duplicates()
)

# 3. Separa citações a OUTROS autores
other_authors_df = (
    df
    .loc[df["author"] != "Mises"]
    [[level, "author"]]
    .drop_duplicates()
)

# 4. Co-citação = mesmo section_id
co_citations = (
    other_authors_df
    .merge(
        mises_df,
        on=level,
        how="inner"
    )
)

# 5. Conta co-citações por autor × parte
author_part_counts = (
    co_citations
    .groupby(["author", "mises_part"])
    .agg(
        co_citation_sections=(level, "nunique")
    )
    .reset_index()
)

# 6. Pivot: matriz autor × partes
author_part_matrix = (
    author_part_counts
    .pivot(
        index="author",
        columns="mises_part",
        values="co_citation_sections"
    )
    .fillna(0)
    .astype(int)
)

# 7. Total de co-citações por autor (todas as partes)
author_part_matrix["total_co_citations"] = author_part_matrix.sum(axis=1)

# 8. Ordena pelo total (decrescente)
author_part_matrix = (
    author_part_matrix
    .sort_values("total_co_citations", ascending=False)
    .reset_index()
)

author_part_matrix.head(100)


mises_part,author,Mises_0,Mises_1,Mises_2,Mises_3,Mises_4,Mises_5,Mises_6,Mises_7,Mises_WHOLE,total_co_citations
0,Hayek,23,101,24,15,115,13,15,7,676,989
1,Rothbard,28,108,11,10,129,2,4,2,283,577
2,Kirzner,8,37,2,8,91,1,11,5,343,506
3,Menger,3,43,10,3,25,1,2,2,126,215
4,Schumpeter,2,20,1,6,31,1,3,0,136,200
5,Lachmann,2,27,7,5,39,0,0,0,113,193
6,Boettke,2,29,5,5,13,5,2,2,121,184
7,Knight,2,17,1,5,9,0,0,0,123,157
8,Foss,0,8,0,0,32,0,2,0,100,142
9,Smith,5,30,7,3,18,2,0,0,56,121
